# 🚀 AI Subtitle Studio - Free Google Colab GPU Backend

Run this notebook on Google Colab with a **free GPU (T4)**.
It will start the Faster-Whisper + OPUS-MT backend and generate a public URL using **localtunnel** or **ngrok**.

### Instructions:
1. Click **Runtime → Change runtime type** → Select **T4 GPU**.
2. Run Cell 1 below to install requirements.
3. Run Cell 2 to start the server and get your **Public Tunnel URL**.
4. Paste the URL into **AI Subtitle Studio** (Settings → Colab Addon).

In [ ]:
# @title 1. Install Dependencies
!pip install -q faster-whisper fastapi uvicorn python-multipart transformers sentencepiece sacremoses
!npm install -g localtunnel
print("\n[✓] Dependencies installed successfully!")

In [ ]:
# @title 2. Start GPU Server & Generate Public URL
import os
import tempfile
import threading
from typing import List
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from pydantic import BaseModel
from faster_whisper import WhisperModel
import uvicorn
from transformers import MarianMTModel, MarianTokenizer

MODEL_SIZE = "large-v3-turbo"  # @param ["tiny", "base", "small", "medium", "large-v3-turbo"]

print(f"[+] Loading Whisper model '{MODEL_SIZE}' onto Colab GPU (CUDA)...")
whisper_model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")
print(f"[✓] Model '{MODEL_SIZE}' ready!")

OPUS_MODEL_MAP = {
    "ja": "Helsinki-NLP/opus-mt-ja-en",
    "ta": "Helsinki-NLP/opus-mt-dra-en",
    "te": "Helsinki-NLP/opus-mt-dra-en",
    "ml": "Helsinki-NLP/opus-mt-dra-en",
    "kn": "Helsinki-NLP/opus-mt-dra-en",
    "hi": "Helsinki-NLP/opus-mt-mul-en",
    "fr": "Helsinki-NLP/opus-mt-fr-en",
    "de": "Helsinki-NLP/opus-mt-de-en",
    "es": "Helsinki-NLP/opus-mt-es-en",
    "zh": "Helsinki-NLP/opus-mt-zh-en",
    "ar": "Helsinki-NLP/opus-mt-ar-en",
    "mul": "Helsinki-NLP/opus-mt-mul-en",
}

loaded_opus = {}
opus_lock = threading.Lock()

def get_opus_model(src_lang: str):
    lang_key = src_lang.lower().split("-")[0]
    hub_name = OPUS_MODEL_MAP.get(lang_key, OPUS_MODEL_MAP.get("mul"))
    with opus_lock:
        if hub_name not in loaded_opus:
            print(f"[+] Loading OPUS-MT '{hub_name}' for language '{src_lang}'...")
            tok = MarianTokenizer.from_pretrained(hub_name)
            mdl = MarianMTModel.from_pretrained(hub_name)
            loaded_opus[hub_name] = (tok, mdl)
        return loaded_opus[hub_name]

def translate_texts_with_opus(texts: List[str], src_lang: str, tgt_lang: str = "en") -> List[str]:
    if not texts or src_lang == tgt_lang:
        return texts
    tok, mdl = get_opus_model(src_lang)
    batch_size = 16
    results = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i : i + batch_size]
        clean_chunk = [t if t.strip() else "." for t in chunk]
        inp = tok(clean_chunk, return_tensors="pt", padding=True, truncation=True, max_length=512)
        out = mdl.generate(**inp, max_length=512)
        decoded = [tok.decode(t, skip_special_tokens=True).strip() for t in out]
        for original, trans in zip(chunk, decoded):
            results.append("" if not original.strip() else trans)
    return results

app = FastAPI(title="Colab Backend")

@app.get("/health")
async def health_check():
    return {"status": "online", "whisper_model": MODEL_SIZE, "device": "Colab CUDA T4"}

class TranslateRequest(BaseModel):
    texts: List[str]
    source_lang: str = "ja"
    target_lang: str = "en"

@app.post("/translate")
async def translate_endpoint(req: TranslateRequest):
    try:
        translated = translate_texts_with_opus(req.texts, req.source_lang, req.target_lang)
        return {"source_lang": req.source_lang, "target_lang": req.target_lang, "translations": translated}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/transcribe")
async def transcribe_audio(
    file: UploadFile = File(...),
    source_lang: str = Form("auto"),
    target_lang: str = Form("en"),
):
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
        content = await file.read()
        tmp.write(content)
        tmp_path = tmp.name

    try:
        lang = None if (not source_lang or source_lang == "auto") else source_lang
        segments_gen, info = whisper_model.transcribe(
            tmp_path,
            language=lang,
            task="transcribe",
            beam_size=5,
            vad_filter=True,
            condition_on_previous_text=False
        )
        raw_segments = list(segments_gen)
        detected_lang = info.language
        original_texts = [s.text.strip() for s in raw_segments]

        translated_texts = []
        effective_src = lang or detected_lang or "ja"
        if target_lang != "original" and target_lang != effective_src:
            try:
                translated_texts = translate_texts_with_opus(original_texts, effective_src, target_lang)
            except Exception:
                translated_texts = original_texts
        else:
            translated_texts = original_texts

        results = []
        for s, orig_text, trans_text in zip(raw_segments, original_texts, translated_texts):
            if orig_text:
                results.append({
                    "startMs": int(s.start * 1000),
                    "endMs": int(s.end * 1000),
                    "original": orig_text,
                    "translation": trans_text,
                    "text": trans_text if target_lang != "original" else orig_text,
                    "confidence": int(round((1.0 - getattr(s, "no_speech_prob", 0.0)) * 100)),
                    "lang": detected_lang
                })
        return {"language": detected_lang, "segments": results}
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

# Start FastAPI in thread
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8001)

t = threading.Thread(target=run_server, daemon=True)
t.start()

# Expose via localtunnel
import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("\n" + "=" * 60)
print(f"  YOUR COLAB PUBLIC IP: {ip}")
print("  (If localtunnel asks for 'Tunnel Password', enter the IP above)")
print("=" * 60 + "\n")

!npx localtunnel --port 8001
